# Z3 的数据类型：表达式、类型、函数声明之间是什么关系

## 这一章要把里子翻出来看

Z3 的那些对象，其实被整整齐齐地组织成了 Python 的类。把这层关系弄明白，以后翻 Z3 的 API 就不会到处乱撞。

- 最上面是 `AstRef`，下面分出 `SortRef`（类型）、`FuncDeclRef`（函数声明）、`ExprRef`（表达式）三支
- 变量和表达式都是 `ExprRef`，`x + y` 和 `x > y` 是一家人
- `sort()` 拿类型，`decl()` 拿函数声明
- 从模型（`ModelRef`）里取值时，类型还会跟着变

前面几章里随口就用的 `Int`、`Bool`、`model[x]`，这一回要把它们里面装的东西看清楚。

In [1]:
import z3
from z3 import *

## Z3 的类关系

先看全貌，这是 Z3 的类继承树。

```
└──Z3PPObject
   ├──AstRef
   │  ├──SortRef
   │  │  ├──BoolSortRef
   │  │  ├──ArithSortRef
   │  │  ├──...
   │  │
   │  ├──FuncDeclRef
   │  └──ExprRef
   │     ├──BoolRef
   │     │  └──QuantifierRef
   │     ├──ArithRef
   │     │  ├──IntNumRef
   │     │  ├──...
   │     ├──...
   ├──ModelRef
   ├──Solver
   └──...
```

`AstRef`（抽象语法树的节点）底下有三个子类。

| 类 | 表示什么 | 例子 |
|---|---|---|
| `SortRef` | 类型（排序） | `Int`, `Bool`, `BitVec(8)` |
| `FuncDeclRef` | 函数或运算符的声明 | `x`（变量）、`>`（比较运算符） |
| `ExprRef` | 表达式 | `x`, `x + y`, `x > y` |

`Int('x')` 和 `x > y` 都是 `ExprRef` 的实例。这话听着抽象，跑一下看真实的类型就明白了。

In [2]:
x, y = Ints('x y')
expr = x > y

print(f'type(x) = {type(x).__name__}')
print(f'type(expr) = {type(expr).__name__}')

type(x) = ArithRef
type(expr) = BoolRef


`x` 是 `ArithRef`（算术表达式），`expr` 是 `BoolRef`（真假表达式），两个都挂在 `ExprRef` 底下。

## ExprRef：看看表达式里面长什么样

表达式是一棵树，有树就有子节点，所以能一层层往下查。

- `num_args()` … 有几个参数
- `arg(i)` … 第 i 个参数是谁

In [3]:
print(f'{x.num_args() = }')
print(f'{expr.num_args() = }')

x.num_args() = 0
expr.num_args() = 2


变量 `x` 一个参数都不带，所以是 0；`x > y` 把 `x` 和 `y` 两个都当参数，所以是 2。

In [4]:
print(f'{expr.arg(0) = }')
print(f'{expr.arg(1) = }')

expr.arg(0) = x
expr.arg(1) = y


`expr = x > y` 的参数果然就是 `x` 和 `y`。一个表达式能拆开、能看清里面装了什么，这就是 `ExprRef` 的本事。

## SortRef：表达式的类型

`sort()` 交回来的是表达式的类型，也就是 `SortRef`。

In [5]:
x_sort = x.sort()
expr_sort = expr.sort()

print(f'{x_sort = }')
print(f'{expr_sort = }')
print(f'type(x_sort) = {type(x_sort).__name__}')
print(f'type(expr_sort) = {type(expr_sort).__name__}')

x_sort = Int
expr_sort = Bool
type(x_sort) = ArithSortRef
type(expr_sort) = BoolSortRef


`x` 是整数，类型自然是 `Int`，这个类型对象的类是 `ArithSortRef`。`expr = x > y` 算出来是真是假，类型就是 `Bool`，类是 `BoolSortRef`。

## FuncDeclRef：函数声明

`decl()` 交回来的是表达式的**函数声明**（`FuncDeclRef`）。在 Z3 内部，变量也好，运算符也好，统统按函数对待。

In [6]:
x_decl = x.decl()
expr_decl = expr.decl()

print(f'{x_decl = }')
print(f'{expr_decl = }')
print(f'type(x_decl) = {type(x_decl).__name__}')
print(f'type(expr_decl) = {type(expr_decl).__name__}')

x_decl = x
expr_decl = >
type(x_decl) = FuncDeclRef
type(expr_decl) = FuncDeclRef


- `x` 是一个不带参数的函数 `x()`，所以它的声明是 `FuncDeclRef`
- `expr = x > y` 里的 `>` 也是函数 `>`，声明同样是 `FuncDeclRef`

手上有了 `FuncDeclRef`，函数的各种信息都能问出来。

| 方法 | 返回的东西 |
|---|---|
| `name()` | 函数的名字 |
| `arity()` | 参数个数 |
| `range()` | 返回值的类型（`SortRef`） |
| `domain(i)` | 第 i 个参数的类型（`SortRef`） |
| `kind()` | Z3 内部给它的编号（整数 ID） |
| `sexpr()` | S 表达式（Lisp 风格的文本） |

In [7]:
print(f'{x_decl.name() = }')
print(f'{x_decl.arity() = }')
print(f'{x_decl.range() = }')

print(f'{expr_decl.name() = }')
print(f'{expr_decl.arity() = }')
print(f'{expr_decl.range() = }')
print(f'{expr_decl.domain(0) = }')
print(f'{expr_decl.domain(1) = }')

x_decl.name() = 'x'
x_decl.arity() = 0
x_decl.range() = Int
expr_decl.name() = '>'
expr_decl.arity() = 2
expr_decl.range() = Bool
expr_decl.domain(0) = Int
expr_decl.domain(1) = Int


变量 `x` 名字叫 `x`，参数 0 个，返回值是 `Int`。`>` 名字叫 `>`，参数 2 个，返回值是 `Bool`，两个参数都是 `Int`。

`kind()` 给的是 Z3 内部的标识符，一个整数。拿它跟 `z3` 模块里的常量对一对，含义就出来了。

In [8]:
print(f'{x_decl.kind() = }   # 用户定义的名称')
print(f'{expr_decl.kind() = }   # 比较运算符 >')

print(f'{z3.Z3_OP_UNINTERPRETED = }')
print(f'{z3.Z3_OP_GT = }')

x_decl.kind() = 49167   # 用户定义的名称
expr_decl.kind() = 517   # 比较运算符 >
z3.Z3_OP_UNINTERPRETED = 49167
z3.Z3_OP_GT = 517


`x_decl.kind()` 和 `z3.Z3_OP_UNINTERPRETED` 是同一个值，说明 `x` 是用户自己定义的、没有被解释的符号。`expr_decl.kind()` 则是 `z3.Z3_OP_GT`，也就是大于运算符。

想知道 Z3 手里那条声明原样长什么，用 `sexpr()` 打出来看。

In [9]:
print(f'{x_decl.sexpr() = }')
print(f'{expr_decl.sexpr() = }')

x_decl.sexpr() = '(declare-fun x () Int)'
expr_decl.sexpr() = '(declare-fun > (Int Int) Bool)'


读出来就是：`x` 声明成 `(declare-fun x () Int)`，`>` 声明成 `(declare-fun > (Int Int) Bool)`。

## 把关系画成图

```{mermaid}
graph
x["Int('x')\nArithRef"]
y["Int('y')\nArithRef"]
expr["x > y\nBoolRef"]
x_decl["x 的声明\nFuncDeclRef"]
expr_decl["> 的声明\nFuncDeclRef"]
int_sort["Int\nArithSortRef"]
bool_sort["Bool\nBoolSortRef"]

x --sort()--> int_sort
y --sort()--> int_sort
expr --sort()--> bool_sort
x --decl()--> x_decl
expr --decl()--> expr_decl
expr --arg(0)--> x
expr --arg(1)--> y
x_decl --range()--> int_sort
expr_decl --range()--> bool_sort
expr_decl --domain(0)--> int_sort
expr_decl --domain(1)--> int_sort
```

## ModelRef：从模型里取值

`Solver.model()` 交回来的是一个 `ModelRef`。这个 `ModelRef` 的 `[]` 很特别，**你塞什么进去，出来的东西就不一样**。

In [10]:
x, y, z = Ints('x y z')
solver = Solver()
solver.add(x > y, y > z, x > 0, z < 5)
solver.check()
m = solver.model()

print(f'type(m) = {type(m).__name__}')
print(f'{m[x] = }  type = {type(m[x]).__name__}')
print(f'{m[0] = }  type = {type(m[0]).__name__}')

type(m) = ModelRef
m[x] = 1  type = IntNumRef
m[0] = z  type = FuncDeclRef


- `m[x]`，塞变量进去 … 拿到这个变量的值，类型是 `IntNumRef`
- `m[0]`，塞整数进去 … 拿到第 0 个变量的**声明**，类型是 `FuncDeclRef`

`IntNumRef` 虽然是 Z3 的整数值，可它不是 Python 的 `int`，要拿去做计算，得先用 `as_long()` 转一道。

In [11]:
print(f'{m[x].as_long() = }')
print(f'type(m[x].as_long()) = {type(m[x].as_long()).__name__}')

# 传入表达式时，返回该表达式的值（BoolRef）
print(f'{m.eval(x > y) = }  type = {type(m.eval(x > y)).__name__}')

m[x].as_long() = 1
type(m[x].as_long()) = int
m.eval(x > y) = True  type = BoolRef


`as_long()` 一转，就是 Python 的整数了。另外 `m.eval(表达式)` 塞的是表达式，还回来的是求值结果（这里是 `x > y` 的真假）。

`IntNumRef` 也有 `decl()`，但它给的不是变量的声明，而是 `Int` 这种值的种类。跟 `x.decl()`（变量 `x` 的声明）完全是两码事。

In [12]:
print(f'{m[x].decl().name() = }')
print(f'{x.decl().name() = }')
print(m[x].decl().eq(x.decl()))

m[x].decl().name() = 'Int'
x.decl().name() = 'x'
False


`m[x].decl()` 的名字是 `'Int'`，`x.decl()` 的名字是 `'x'`。调的明明是同一个 `decl()`，可对着谁调用，意思就变了。

## 继承关系对一对

`type(obj).mro()` 能把一个类的祖先一路列出来。

In [13]:
print(type(x).mro())
print(type(x.decl()).mro())
print(type(m).mro())

[<class 'z3.z3.ArithRef'>, <class 'z3.z3.ExprRef'>, <class 'z3.z3.AstRef'>, <class 'z3.z3.Z3PPObject'>, <class 'object'>]
[<class 'z3.z3.FuncDeclRef'>, <class 'z3.z3.AstRef'>, <class 'z3.z3.Z3PPObject'>, <class 'object'>]
[<class 'z3.z3.ModelRef'>, <class 'z3.z3.Z3PPObject'>, <class 'object'>]


`ArithRef` 一路往上：`ExprRef` → `AstRef` → `Z3PPObject`；`FuncDeclRef` 则直接挂在 `AstRef` 下面。跟前面那张类图对得上。

再把 `AstRef` 的子类实际列一遍。

In [14]:
print([cls.__name__ for cls in z3.AstRef.__subclasses__()])


def print_tree(cls, indent=0, depth=2):
    print("  " * indent + cls.__name__)
    if depth == 0:
        return
    for sub in cls.__subclasses__():
        print_tree(sub, indent + 1, depth - 1)


print()
print_tree(z3.SortRef, depth=1)

['SortRef', 'FuncDeclRef', 'ExprRef']

SortRef
  TypeVarRef
  BoolSortRef
  ArithSortRef
  BitVecSortRef
  ArraySortRef
  FiniteSetSortRef
  DatatypeSortRef
  FiniteDomainSortRef
  FPSortRef
  FPRMSortRef
  SeqSortRef
  CharSortRef
  ReSortRef


`SortRef` 底下排着一串按类型分的类：`BoolSortRef`、`ArithSortRef`、`BitVecSortRef`，等等。前面几章用过的 `BitVecSort`、`RealSort`、`IntSort`，也全是 `SortRef` 这一家的。

## 收成一页

| 想知道什么 | 用哪个 | 返回的类型 |
|---|---|---|
| 变量或表达式的类型 | `expr.sort()` | `SortRef` |
| 表达式子节点的个数 | `expr.num_args()` | `int` |
| 表达式的第 i 个子节点 | `expr.arg(i)` | `ExprRef` |
| 函数声明 | `expr.decl()` | `FuncDeclRef` |
| 函数的名称、参数、类型 | `name()`, `arity()`, `range()`, `domain(i)` | 字符串、数、`SortRef` |
| 模型的值 | `m[x]` | `IntNumRef` |
| 转成 Python 整数 | `.as_long()` | `int` |

- `Int`、`Bool`、`x > y`，说到底都是 `ExprRef`（表达式）
- 类型归 `SortRef`，函数声明归 `FuncDeclRef`，表达式归 `ExprRef`，三个都在 `AstRef` 门下
- `model[0]` 给的是声明，`model[x]` 给的是值。同样的 `[]`，两种意思